In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

data = [
    ("C1", "2025-01-01", 100),
    ("C1", "2025-01-05", 200),
    ("C1", "2025-01-10", 150),
    ("C1", "2025-01-11", 150),
    ("C2", "2025-01-03", 300),
    ("C2", "2025-01-08", 100)
]

df = spark.createDataFrame(data, ["customer_id", "order_date", "amount"]) \
          .withColumn("order_date", to_date("order_date"))

df.show()


+-----------+----------+------+
|customer_id|order_date|amount|
+-----------+----------+------+
|         C1|2025-01-01|   100|
|         C1|2025-01-05|   200|
|         C1|2025-01-10|   150|
|         C1|2025-01-11|   150|
|         C2|2025-01-03|   300|
|         C2|2025-01-08|   100|
+-----------+----------+------+



## Specify Window

In [8]:
w = Window.partitionBy("customer_id").orderBy("order_date")


### Row Number

In [9]:
df.withColumn("row_num", row_number().over(w)).show()


+-----------+----------+------+-------+
|customer_id|order_date|amount|row_num|
+-----------+----------+------+-------+
|         C1|2025-01-01|   100|      1|
|         C1|2025-01-05|   200|      2|
|         C1|2025-01-10|   150|      3|
|         C1|2025-01-11|   150|      4|
|         C2|2025-01-03|   300|      1|
|         C2|2025-01-08|   100|      2|
+-----------+----------+------+-------+



### Rank vs Dense Rank

In [10]:
w_amount = Window.partitionBy("customer_id").orderBy(desc("amount"))

df.withColumn("rank", rank().over(w_amount)) \
  .withColumn("dense_rank", dense_rank().over(w_amount)) \
  .show()


+-----------+----------+------+----+----------+
|customer_id|order_date|amount|rank|dense_rank|
+-----------+----------+------+----+----------+
|         C1|2025-01-05|   200|   1|         1|
|         C1|2025-01-10|   150|   2|         2|
|         C1|2025-01-11|   150|   2|         2|
|         C1|2025-01-01|   100|   4|         3|
|         C2|2025-01-03|   300|   1|         1|
|         C2|2025-01-08|   100|   2|         2|
+-----------+----------+------+----+----------+



### Running Total

In [11]:
df.withColumn(
    "running_total",
    sum("amount").over(w)
).show()


+-----------+----------+------+-------------+
|customer_id|order_date|amount|running_total|
+-----------+----------+------+-------------+
|         C1|2025-01-01|   100|          100|
|         C1|2025-01-05|   200|          300|
|         C1|2025-01-10|   150|          450|
|         C1|2025-01-11|   150|          600|
|         C2|2025-01-03|   300|          300|
|         C2|2025-01-08|   100|          400|
+-----------+----------+------+-------------+



### Rolling Window (Last 2 Orders)

In [12]:
w_rolling = w.rowsBetween(-1, 0)

df.withColumn(
    "last_2_sum",
    sum("amount").over(w_rolling)
).show()


+-----------+----------+------+----------+
|customer_id|order_date|amount|last_2_sum|
+-----------+----------+------+----------+
|         C1|2025-01-01|   100|       100|
|         C1|2025-01-05|   200|       300|
|         C1|2025-01-10|   150|       350|
|         C1|2025-01-11|   150|       300|
|         C2|2025-01-03|   300|       300|
|         C2|2025-01-08|   100|       400|
+-----------+----------+------+----------+



## Lag / Lead

### Previous Order Amount

In [13]:
df.withColumn("prev_amount", lag("amount").over(w)).show()


+-----------+----------+------+-----------+
|customer_id|order_date|amount|prev_amount|
+-----------+----------+------+-----------+
|         C1|2025-01-01|   100|       NULL|
|         C1|2025-01-05|   200|        100|
|         C1|2025-01-10|   150|        200|
|         C1|2025-01-11|   150|        150|
|         C2|2025-01-03|   300|       NULL|
|         C2|2025-01-08|   100|        300|
+-----------+----------+------+-----------+



### Amount Change

In [14]:
df.withColumn(
    "amount_diff",
    col("amount") - lag("amount").over(w)
).show()


+-----------+----------+------+-----------+
|customer_id|order_date|amount|amount_diff|
+-----------+----------+------+-----------+
|         C1|2025-01-01|   100|       NULL|
|         C1|2025-01-05|   200|        100|
|         C1|2025-01-10|   150|        -50|
|         C1|2025-01-11|   150|          0|
|         C2|2025-01-03|   300|       NULL|
|         C2|2025-01-08|   100|       -200|
+-----------+----------+------+-----------+



## Window for Deduplication

In [17]:
# Get last order for each customer
df_latest = (
    df.withColumn("rn", row_number().over(
        Window.partitionBy("customer_id").orderBy(desc("order_date"))
    ))
    .filter(col("rn") == 1)
)

df_latest.show()


+-----------+----------+------+---+
|customer_id|order_date|amount| rn|
+-----------+----------+------+---+
|         C1|2025-01-11|   150|  1|
|         C2|2025-01-08|   100|  1|
+-----------+----------+------+---+

